# 04 - Datenanalyse & Beantwortung der Fragestellungen

**Projekt:** Polymarket Reddit Sentiment  
**Kurs:** Data Wrangling & Engineering (FHNW)

> Hinweis: Dieses Notebook nutzt bevorzugt die finalen CSVs aus `../data/`. Der aktuelle, benotungsrelevante Stand steht in `../reports/FINAL_REPORT.pdf`; die Zelloutputs wurden entfernt, damit keine alten Runs mit dem finalen Report kollidieren.

## Fragestellungen

1. **F1:** Korrelieren Reddit-Sentiment-Scores mit Polymarket-Wahrscheinlichkeiten?
2. **F1b:** Unterscheidet sich diese Korrelation nach Marktkategorie?
3. **F2:** Unterscheidet sich das Sentiment signifikant zwischen Subreddits?
4. **F3:** Gibt es zeitliche Muster im Reddit-Sentiment?
5. **F4:** Erklaert Stance Detection die Marktwahrscheinlichkeit besser als reines Sentiment?

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src import reddit, polymarket, sentiment, market_metadata

print('Libraries geladen.')


In [ ]:
import os

# Finalen Bulk-Run bevorzugen. Alte Notebook-CSV ist nur Fallback fuer Exploration.
POSTS_BULK = '../data/posts_per_market.csv'
POSTS_OLD  = '../data/reddit_clean.csv'
PAIRS_BULK = '../data/correlation_pairs_bulk.csv'

if os.path.exists(POSTS_BULK):
    posts_df = pd.read_csv(POSTS_BULK, parse_dates=['created_utc'])
    if 'sentiment_label' not in posts_df.columns:
        posts_df['sentiment_label'] = posts_df['compound'].apply(
            lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
        )
    print(f'Posts geladen (finaler Bulk-Run): {posts_df.shape}')
    print(f"Maerkte: {posts_df['market_question'].nunique()}")
    print(f"Subreddits: {sorted(posts_df['subreddit'].dropna().unique())}")
elif os.path.exists(POSTS_OLD):
    posts_df = pd.read_csv(POSTS_OLD, parse_dates=['created_utc'])
    if 'sentiment_label' not in posts_df.columns:
        posts_df['sentiment_label'] = posts_df['compound'].apply(
            lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
        )
    print(f'Posts geladen (explorativer Fallback reddit_clean.csv): {posts_df.shape}')
else:
    raw = reddit.get_posts('Bitcoin', ['investing', 'stocks', 'worldnews', 'CryptoCurrency'], 100)
    posts_df = sentiment.analyze(raw)
    posts_df['sentiment_label'] = posts_df['compound'].apply(
        lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
    )
    print(f'Posts live geladen (explorativer Fallback): {posts_df.shape}')

if os.path.exists(PAIRS_BULK):
    markets_df = pd.read_csv(PAIRS_BULK)
    print(f'Maerkte geladen (finaler Bulk-Run): {markets_df.shape}')
elif os.path.exists('../data/polymarket_clean.csv'):
    markets_df = pd.read_csv('../data/polymarket_clean.csv')
    print(f'Polymarket (bereinigt, explorativ): {markets_df.shape}')
else:
    markets_df = pd.DataFrame()


---
## F1: Korrelation zwischen Reddit-Sentiment und Polymarket-Wahrscheinlichkeit

### Methode
Statt alle Posts pauschal zu vergleichen, gehen wir **pro Markt** vor:

```
Für jeden Polymarket-Markt:
  1. Keywords aus der Frage extrahieren
  2. Reddit gezielt nach diesen Keywords durchsuchen
  3. Sentiment der gefundenen Posts berechnen
  4. Paar speichern: (Markt-Wahrscheinlichkeit, Ø Reddit-Sentiment)

Danach:
  → Pearson-Korrelation (lineare Beziehung)
  → Spearman-Korrelation (Rangkorrelation, robust gegen Ausreisser)
  → Scatter-Plot mit Regressionslinie
```

In [ ]:
import time

# Konfiguration fuer einen optionalen Neu-Run aus dem Notebook.
# Fuer die Abgabe ist `python run_bulk.py` der reproduzierbare Hauptlauf.
MAX_MARKETS        = 30
POSTS_PER_MARKET   = 25
INCLUDE_COMMENTS   = False
COMMENT_LIMIT      = 0
SEMANTIC_THRESHOLD = 0.20
SENTIMENT_MODEL    = sentiment.MODEL_ROBERTA
SUBREDDITS = ["politics", "worldnews", "stocks", "investing", "news",
              "Economics", "geopolitics"]

extract_keywords = market_metadata.extract_keywords

try:
    _live = polymarket.get_markets(limit=MAX_MARKETS)
    if not _live.empty and 'probability' in _live.columns:
        sample_markets = market_metadata.filter_relevant_markets(_live, MAX_MARKETS).reset_index(drop=True)
        print(f'Polymarket live: {len(sample_markets)} Maerkte geladen.')
    else:
        raise ValueError('Leere API-Antwort')
except Exception:
    from run_bulk import SAMPLE_DATA
    print('Polymarket API nicht verfuegbar -> markierter Demo-Fallback fuer Notebook-Exploration.')
    sample_markets = pd.DataFrame(SAMPLE_DATA).head(MAX_MARKETS)

print(f'Konfiguration: {len(sample_markets)} Maerkte, {POSTS_PER_MARKET} Posts/Markt, Modell={SENTIMENT_MODEL}')
sample_markets.head(5)


In [ ]:
# ── Sentiment-Modell Konfiguration ───────────────────────────────────────────
#
# Modell wählen:
#   sentiment.MODEL_VADER           – schnell, kein Download (~sofort)
#   sentiment.MODEL_FINBERT         – Finanztexte, ~440 MB, pip install transformers torch
#   sentiment.MODEL_ROBERTA         – Social Media, ~500 MB, pip install transformers torch
#
SENTIMENT_MODEL  = sentiment.MODEL_FINBERT  # FinBERT: domänenspezifisch für Finanztexte
INCLUDE_COMMENTS = True                     # Kommentare zu Posts mitholen
COMMENT_LIMIT    = 10                       # Kommentare pro Post

print(f'Sentiment-Modell : {SENTIMENT_MODEL}')
print(f'Mit Kommentaren  : {INCLUDE_COMMENTS} (max. {COMMENT_LIMIT} pro Post)')
print(f'Sem. Filterung   : threshold={SEMANTIC_THRESHOLD} (sentence-transformers)')

try:
    import transformers
    print(f'transformers {transformers.__version__} verfügbar.')
except ImportError:
    print('FEHLER: pip install transformers torch')

try:
    import sentence_transformers
    print(f'sentence-transformers {sentence_transformers.__version__} verfügbar.')
except ImportError:
    print('FEHLER: pip install sentence-transformers')


In [ ]:
# ── Pro Markt: Posts holen, semantisch filtern & Sentiment berechnen ──────────
import os

CSV_PATH = '../data/correlation_pairs_bulk.csv'  # finaler Bulk-Run, Ziel: 25-30 auswertbare Maerkte

if os.path.exists(CSV_PATH):
    pairs_df = pd.read_csv(CSV_PATH)
    print(f'Lade gespeicherte Paare: {len(pairs_df)} Maerkte aus {CSV_PATH}')
    print(pairs_df[['question','probability','mean_compound','n_posts','category']].to_string(index=False))
else:
    pairs = []

    for i, row in sample_markets.iterrows():
        question = row['question']
        prob     = row['probability']
        category = row.get('category', 'Unknown')
        keywords = extract_keywords(question)

        try:
            # 1. Reddit-Posts per Keyword-Suche holen
            raw = reddit.get_posts(
                keywords,
                SUBREDDITS,
                POSTS_PER_MARKET,
                include_comments=INCLUDE_COMMENTS,
                comment_limit=COMMENT_LIMIT,
            )
            if raw.empty:
                print(f'  SKIP [{keywords}] keine Posts')
                continue

            # 2. Semantische Filterung: nur Posts die zur Frage passen
            raw = sentiment.semantic_filter(raw, question, threshold=SEMANTIC_THRESHOLD)
            if len(raw) < 3:
                print(f'  SKIP [{keywords}] zu wenige relevante Posts nach Filterung')
                continue

            n_posts    = (raw['content_type'] == 'post').sum()    if 'content_type' in raw.columns else len(raw)
            n_comments = (raw['content_type'] == 'comment').sum() if 'content_type' in raw.columns else 0

            # 3. Sentiment mit konfiguriertem Modell
            scored = sentiment.analyze(raw, model=SENTIMENT_MODEL)
            mean_s = scored['compound'].mean()
            weights = np.log1p(scored['score'].clip(lower=0).fillna(0).values)
            weighted_s = np.average(scored['compound'].values, weights=weights) if weights.sum() > 0 else mean_s

            pairs.append({
                'question':          question,
                'probability':       prob,
                'mean_compound':     mean_s,
                'weighted_compound': weighted_s,
                'n_posts':           n_posts,
                'n_comments':        n_comments,
                'n_total':           len(scored),
                'keywords':          keywords,
                'category':          category,
            })

            print(f'  [{len(pairs):>2}] p={prob:.2f}  mean={mean_s:+.3f}  wtd={weighted_s:+.3f}'
                  f'  posts={n_posts}  comments={n_comments}  [{keywords}]')

            time.sleep(3.0)

        except Exception as e:
            print(f'  Fehler bei "{question[:40]}": {e}')

    pairs_df = pd.DataFrame(pairs)
    os.makedirs('../data', exist_ok=True)
    pairs_df.to_csv(CSV_PATH, index=False)
    print(f'\nGespeichert: {CSV_PATH}')

print(f'\nPaare gesamt: {len(pairs_df)}')
if not pairs_df.empty:
    print(f'Ø Posts/Markt:      {pairs_df["n_posts"].mean():.0f}')
    print(f'Ø Kommentare/Markt: {pairs_df["n_comments"].mean():.0f}')
pairs_df.head()


In [ ]:
# Vorzeichen-Korrektur: zentrale Polarity-Logik aus src/market_metadata.py
# Positive Stimmung bedeutet bei negativ gerahmten Fragen nicht automatisch hoehere Eintrittswahrscheinlichkeit.
question_polarity = market_metadata.question_polarity

pairs_df['polarity']          = pairs_df['question'].apply(question_polarity)
pairs_df['adjusted_compound'] = pairs_df['mean_compound']     * pairs_df['polarity']
pairs_df['adjusted_weighted'] = pairs_df['weighted_compound'] * pairs_df['polarity']

n_neg = (pairs_df['polarity'] == -1).sum()
n_pos = (pairs_df['polarity'] == +1).sum()
print(f'Polaritaet: {n_pos} positiv-gerahmte, {n_neg} negativ-gerahmte Fragen')
print()
neg_rows = pairs_df[pairs_df['polarity'] == -1][['question', 'polarity']]
if not neg_rows.empty:
    print('Negativ-gerahmt (Vorzeichen invertiert):')
    for q in neg_rows['question']:
        print(' ', q[:70])


In [ ]:
# ── Ranglisten-Tabelle ───────────────────────────────────────────────────────
if len(pairs_df) >= 4:
    display_df = pairs_df[['question', 'probability', 'mean_compound', 'n_posts', 'category']].copy()
    display_df = display_df.sort_values('probability', ascending=False)
    display_df['probability']   = display_df['probability'].map('{:.1%}'.format)
    display_df['mean_compound'] = display_df['mean_compound'].map('{:+.3f}'.format)
    display_df.columns = ['Markt-Frage', 'Polymarket %', 'Reddit-Sentiment', 'Posts', 'Kategorie']
    print('Alle Paare sortiert nach Polymarket-Wahrscheinlichkeit:')
    print(display_df.to_string(index=False))

    # Pairs speichern
    import os
    os.makedirs('../data', exist_ok=True)
    pairs_df.to_csv('../data/correlation_pairs.csv', index=False)
    print('\nGespeichert: data/correlation_pairs.csv')

In [ ]:
# ── Korrelation: ungewichtet UND gewichtet im Vergleich ──────────────────────
if len(pairs_df) < 4:
    print('Zu wenige Datenpunkte (min. 4 nötig).')
else:
    prob_vals = pairs_df['probability'].values
    mean_vals = pairs_df['adjusted_compound'].values
    wtd_vals  = pairs_df['adjusted_weighted'].values

    results = {}
    for label, y in [('Ungewichtet (polarity-adj.)', mean_vals), ('Gewichtet (polarity-adj.)', wtd_vals)]:
        r_p, p_p = stats.pearsonr(prob_vals, y)
        r_s, p_s = stats.spearmanr(prob_vals, y)
        results[label] = dict(r_pearson=r_p, p_pearson=p_p, r_spearman=r_s, p_spearman=p_s)

    print('=' * 60)
    print('KORRELATIONSVERGLEICH: ungewichtet vs. gewichtet')
    print('=' * 60)
    for label, res in results.items():
        sig_p = '*' if res['p_pearson']  < 0.05 else ' '
        sig_s = '*' if res['p_spearman'] < 0.05 else ' '
        print(f'\n{label}:')
        print(f'  Pearson  r = {res["r_pearson"]:+.4f}   p = {res["p_pearson"]:.4f} {sig_p}')
        print(f'  Spearman ρ = {res["r_spearman"]:+.4f}   p = {res["p_spearman"]:.4f} {sig_s}')
        absR = abs(res['r_spearman'])
        strength = 'stark' if absR >= 0.7 else 'mittel' if absR >= 0.4 else 'schwach' if absR >= 0.2 else 'keine/kaum'
        print(f'  → {strength} Korrelation')

    # ── Scatter-Plot: zwei Panels nebeneinander ───────────────────────────────
    categories = pairs_df['category'].fillna('Other').unique()
    palette    = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22','#95a5a6']
    cat_color  = {c: palette[i % len(palette)] for i, c in enumerate(categories)}

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

    for ax, (col, title) in zip(axes, [
        ('mean_compound',     'Ungewichtetes Sentiment'),
        ('weighted_compound', 'Gewichtetes Sentiment (log-Upvotes)'),
    ]):
        y = pairs_df[col].values
        slope, intercept, r_val, p_val, _ = stats.linregress(prob_vals, y)
        x_line = np.linspace(prob_vals.min() - 0.04, prob_vals.max() + 0.04, 100)

        for cat in categories:
            sub = pairs_df[pairs_df['category'].fillna('Other') == cat]
            ax.scatter(sub['probability'], sub[col],
                       s=sub['n_posts'] * 2 + 30,
                       color=cat_color[cat], alpha=0.75,
                       edgecolors='white', linewidths=0.7,
                       label=cat, zorder=3)

        ax.plot(x_line, slope * x_line + intercept,
                color='black', linewidth=1.5, linestyle='--',
                label=f'Regression  r={r_val:+.3f}  p={p_val:.3f}', zorder=4)

        ax.axhline(0,    color='gray',  lw=0.7, ls=':')
        ax.axhline(0.05, color='green', lw=0.5, ls=':', alpha=0.5)
        ax.axhline(-0.05,color='red',   lw=0.5, ls=':', alpha=0.5)
        ax.axvline(0.5,  color='gray',  lw=0.7, ls=':', label='50% Markt')

        for _, row in pairs_df.iterrows():
            short = row['keywords'].split()[0] if row['keywords'] else ''
            ax.annotate(short, (row['probability'], row[col]),
                        textcoords='offset points', xytext=(4, 2),
                        fontsize=6.5, alpha=0.65)

        ax.set_xlabel('Polymarket Wahrscheinlichkeit', fontsize=10)
        ax.set_ylabel('Reddit Compound Score', fontsize=10)
        ax.set_title(f'{title}\nPearson r={r_val:+.3f}  |  n={len(pairs_df)} Märkte', fontsize=10)
        ax.legend(fontsize=7, loc='best', ncol=2)
        ax.grid(True, alpha=0.18)

    plt.suptitle('Polymarket-Wahrscheinlichkeit vs. Reddit-Sentiment\nPunktgrösse = Anzahl Posts',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig('../notebooks/analyse_f1_korrelation.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('Gespeichert: notebooks/analyse_f1_korrelation.png')

    # Für Fazit speichern
    r_pearson  = results['Gewichtet (polarity-adj.)']['r_pearson']
    p_pearson  = results['Gewichtet (polarity-adj.)']['p_pearson']
    r_spearman = results['Gewichtet (polarity-adj.)']['r_spearman']
    p_spearman = results['Gewichtet (polarity-adj.)']['p_spearman']

In [ ]:
# ── Richtungsübereinstimmung: Stimmt die Richtung überein? ───────────────────
# Polymarket > 50%  →  "Markt erwartet JA"
# Reddit compound > 0  →  "Reddit-Stimmung positiv"
# Übereinstimmung = beide zeigen dieselbe Richtung

if len(pairs_df) >= 4:
    pairs_df['market_optimistic']    = (pairs_df['probability'] > 0.5).astype(int)
    pairs_df['sentiment_optimistic'] = (pairs_df['weighted_compound'] > 0).astype(int)
    accuracy = (pairs_df['market_optimistic'] == pairs_df['sentiment_optimistic']).mean()

    agree   = (pairs_df['market_optimistic'] == pairs_df['sentiment_optimistic']).sum()
    disagree = len(pairs_df) - agree

    print(f'Richtungsübereinstimmung: {accuracy:.1%}  ({agree} von {len(pairs_df)} Märkten)')
    print(f'  Übereinstimmung: {agree}   Widerspruch: {disagree}')
    print()

    # Aufschlüsselung nach Kategorie
    pairs_df['match'] = (pairs_df['market_optimistic'] == pairs_df['sentiment_optimistic'])
    cat_acc = pairs_df.groupby('category')['match'].mean().sort_values(ascending=False)
    print('Richtungsübereinstimmung nach Kategorie:')
    for cat, acc in cat_acc.items():
        bar = '#' * int(acc * 20)
        print(f'  {cat:<14} {acc:.0%}  {bar}')

    # Bar-Chart
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Links: Gesamt-Donut
    ax = axes[0]
    ax.pie([agree, disagree], labels=[f'Übereinstimmung\n{agree}', f'Widerspruch\n{disagree}'],
           colors=['#2ecc71', '#e74c3c'], autopct='%1.0f%%', startangle=90,
           wedgeprops=dict(width=0.5))
    ax.set_title(f'Richtungsübereinstimmung gesamt\n{accuracy:.1%} korrekt  (n={len(pairs_df)})')

    # Rechts: Nach Kategorie
    ax = axes[1]
    cats = cat_acc.index.tolist()
    vals = cat_acc.values
    colors_bar = ['#2ecc71' if v >= 0.6 else '#f39c12' if v >= 0.4 else '#e74c3c' for v in vals]
    ax.barh(cats, vals, color=colors_bar, edgecolor='white')
    ax.axvline(0.5, color='gray', linestyle='--', lw=1, label='Zufall (50%)')
    ax.set_xlim(0, 1)
    ax.set_xlabel('Richtungsübereinstimmung')
    ax.set_title('Übereinstimmung nach Kategorie')
    ax.legend(fontsize=8)
    for i, v in enumerate(vals):
        ax.text(v + 0.01, i, f'{v:.0%}', va='center', fontsize=9)
    ax.grid(axis='x', alpha=0.2)

    plt.tight_layout()
    plt.savefig('../notebooks/analyse_richtung.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Gespeichert: notebooks/analyse_richtung.png')


---
## F1b: Kategorie-Analyse - Welche Markttypen korrelieren am staerksten?

**Fragestellung:** Variiert die Korrelation zwischen Reddit-Sentiment und Polymarket-Wahrscheinlichkeit
je nach Markt-Kategorie? Die finale Taxonomie kommt aus `src/market_metadata.py`.

**Methodik:**
1. Finalen Bulk-Run aus `data/correlation_pairs_bulk.csv` laden
2. Kategorien aus stabiler Taxonomie verwenden oder aus der Marktfrage inferieren
3. Pearson r pro Kategorie berechnen (Probability vs. adjusted_weighted)
4. Kleine Gruppen als explorativ markieren

> Hinweis: Echte Lag-Analyse erfordert wiederholte Polymarket-Snapshots. Der finale Datensatz
> ist ein Querschnitt und wird deshalb nicht kausal interpretiert.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

infer_category = market_metadata.infer_category
BULK_CSV = '../data/correlation_pairs_bulk.csv'
print('=== F1b: Korrelation nach Markt-Kategorie ===')

if not os.path.exists(BULK_CSV):
    print('Keine Bulk-Daten. Bitte: python run_bulk.py')
else:
    df_bulk = pd.read_csv(BULK_CSV)
    if 'polarity' not in df_bulk.columns:
        df_bulk['polarity'] = df_bulk['question'].apply(market_metadata.question_polarity)
    if 'adjusted_compound' not in df_bulk.columns:
        df_bulk['adjusted_compound'] = df_bulk['mean_compound'] * df_bulk['polarity']
    if 'adjusted_weighted' not in df_bulk.columns:
        df_bulk['adjusted_weighted'] = df_bulk['weighted_compound'] * df_bulk['polarity']
    if 'category' in df_bulk.columns:
        df_bulk['category_inferred'] = df_bulk['category'].fillna(df_bulk['question'].apply(infer_category))
    else:
        df_bulk['category_inferred'] = df_bulk['question'].apply(infer_category)

    print(f'Datensatz: {len(df_bulk)} Maerkte')
    print('Kategorien:', df_bulk['category_inferred'].value_counts().to_dict())

    cat_results = []
    for cat, grp in df_bulk.dropna(subset=['probability', 'adjusted_weighted']).groupby('category_inferred'):
        if len(grp) < 3:
            print(f'  {cat}: nur {len(grp)} Maerkte - explorativ/uebersprungen')
            continue
        r, p = stats.pearsonr(grp['probability'], grp['adjusted_weighted'])
        cat_results.append({'category': cat, 'n': len(grp), 'pearson_r': r, 'p_value': p})

    cat_results_df = pd.DataFrame(cat_results).sort_values('pearson_r') if cat_results else pd.DataFrame()
    display(cat_results_df)

    if not cat_results_df.empty:
        colors = ['#ef4444' if v < 0 else '#22c55e' for v in cat_results_df['pearson_r']]
        ax = cat_results_df.plot.barh(x='category', y='pearson_r', color=colors, legend=False, figsize=(9, 4.8))
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlim(-1, 1)
        ax.set_xlabel('Pearson r')
        ax.set_title('F1b: Korrelation nach Markt-Kategorie')
        for i, row in cat_results_df.reset_index(drop=True).iterrows():
            ax.text(row['pearson_r'] + (0.03 if row['pearson_r'] >= 0 else -0.03), i,
                    f"n={int(row['n'])}, p={row['p_value']:.2f}",
                    va='center', ha='left' if row['pearson_r'] >= 0 else 'right', fontsize=8)
        plt.tight_layout()
        plt.show()


In [ ]:
# ── Z-Score-Normalisierung: FinBERT und RoBERTa auf gleiche Skala ──────────
# Problem: FinBERT (alle negativ -0.09..-0.32) vs RoBERTa (-0.18..+0.19)
# Lösung: z-Score pro Modell-Gruppe → Werte direkt vergleichbar
if 'combined_df' in dir() and not combined_df.empty and 'model' in combined_df.columns:
    combined_df['compound_norm'] = (
        combined_df.groupby('model')['mean_compound']
        .transform(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else x * 0)
    )
    r_norm, p_norm = stats.pearsonr(combined_df['probability'], combined_df['compound_norm'])
    r_raw,  p_raw  = stats.pearsonr(combined_df['probability'], combined_df['mean_compound'])
    print('Z-Score-Normalisierung:')
    print(f'  Pearson r (roh):         {r_raw:+.4f}  (p={p_raw:.4f})')
    print(f'  Pearson r (normalisiert): {r_norm:+.4f}  (p={p_norm:.4f})')
    sig = 'signifikant (p<0.05)' if p_norm < 0.05 else 'nicht signifikant'
    print(f'  -> {sig}')
    print()
    print('Modell-Statistiken (vor Normalisierung):')
    print(combined_df.groupby('model')['mean_compound'].describe().round(3).to_string())

# ── Kombinierte Korrelation: FinBERT + VADER (30+ Märkte) ─────────────
if 'combined_df' in dir() and not combined_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Scatter: alle Märkte, eingefärbt nach Modell
    colors_map = {'finbert': '#3498db', 'vader': '#e67e22', 'roberta': '#9b59b6'}
    for model, grp in combined_df.groupby('model'):
        axes[0].scatter(grp['probability'], grp['mean_compound'],
                        alpha=0.7, label=model.upper(),
                        color=colors_map.get(model, 'gray'), s=60)
    # Gesamttrendlinie
    m, b = np.polyfit(combined_df['probability'], combined_df['mean_compound'], 1)
    xs = np.linspace(0, 1, 100)
    axes[0].plot(xs, m*xs+b, 'k--', alpha=0.5, label=f'Trend (r={r_all:+.3f})')
    axes[0].axhline(0, color='gray', lw=0.8, ls=':')
    axes[0].axvline(0.5, color='gray', lw=0.8, ls=':')
    axes[0].set_xlabel('Polymarket-Wahrscheinlichkeit')
    axes[0].set_ylabel('Ø Sentiment (mean compound)')
    axes[0].set_title(f'Sentiment vs. Marktpreis\n(n={len(combined_df)}, Pearson r={r_all:+.3f}, p={p_all:.3f})')
    axes[0].legend()

    # Boxplot nach Kategorie
    # Fehlende Kategorie-Labels auffüllen (Polymarket API gibt oft leere Strings)
    _combined_plot = combined_df.copy()
    _combined_plot['category'] = _combined_plot['category'].fillna('Unknown').replace('', 'Unknown')
    cats_dict = {cat: grp['mean_compound'].tolist() for cat, grp in _combined_plot.groupby('category')}
    if cats_dict:
        axes[1].boxplot(list(cats_dict.values()), tick_labels=list(cats_dict.keys()))
    else:
        axes[1].text(0.5, 0.5, 'Keine Kategorien verfügbar', ha='center', transform=axes[1].transAxes)
    axes[1].axhline(0, color='gray', lw=0.8, ls='--')
    axes[1].set_title('Sentiment nach Kategorie (alle Märkte)')
    axes[1].set_ylabel('Ø Compound')
    plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig('analyse_combined_korrelation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gespeichert: notebooks/analyse_combined_korrelation.png')
    print(f'\nKombinierter Datensatz ({len(combined_df)} Maerkte):')
    r_s, p_s = stats.spearmanr(combined_df['probability'], combined_df['mean_compound'])
    print(f'  Pearson  r = {r_all:+.4f}   p = {p_all:.4f}')
    print(f'  Spearman r = {r_s:+.4f}   p = {p_s:.4f}')
    sig = 'signifikant (p<0.05)' if p_all < 0.05 else 'nicht signifikant'
    print(f'  -> {sig}')


---
## F2: Unterscheidet sich das Sentiment zwischen verschiedenen Subreddits?

In [ ]:
sub_stats = posts_df.groupby('subreddit').agg(
    n=('compound', 'count'),
    mean=('compound', 'mean'),
    std=('compound', 'std'),
    median=('compound', 'median'),
    positive_pct=('sentiment_label', lambda x: (x == 'positive').mean() * 100)
).round(4).sort_values('mean', ascending=False)

print('Sentiment-Statistik nach Subreddit:')
print(sub_stats.to_string())

In [ ]:
# Statistischer Test: Kruskal-Wallis (nicht-parametrisch, da Compound nicht normalverteilt)
groups = [group['compound'].values for _, group in posts_df.groupby('subreddit') if len(group) >= 5]

if len(groups) >= 2:
    h_stat, p_value = stats.kruskal(*groups)
    print(f'Kruskal-Wallis Test:')
    print(f'  H-Statistik: {h_stat:.4f}')
    print(f'  p-Wert:      {p_value:.6f}')
    if p_value < 0.05:
        print('  → Signifikanter Unterschied zwischen Subreddits (p < 0.05)')
    else:
        print('  → Kein signifikanter Unterschied (p >= 0.05)')
else:
    print('Zu wenige Gruppen für statistischen Test (mindestens 2 Subreddits mit je 5+ Posts).')

In [ ]:
# ── F2: Box-Plot Sentiment nach Subreddit (Matplotlib) ───────────────────────
subs    = sub_stats.index.tolist()
palette = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']
colors  = {s: palette[i % len(palette)] for i, s in enumerate(subs)}

fig, ax = plt.subplots(figsize=(10, 5))

for i, sub in enumerate(subs):
    vals = posts_df[posts_df['subreddit'] == sub]['compound'].values
    bp = ax.boxplot(vals, positions=[i], widths=0.5,
                    patch_artist=True, notch=False,
                    boxprops=dict(facecolor=colors[sub], alpha=0.6),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', markersize=4, alpha=0.5,
                                    markerfacecolor=colors[sub]))
    # Einzelpunkte als Jitter
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax.scatter([i + j for j in jitter], vals,
               color=colors[sub], alpha=0.55, s=20, zorder=3)

ax.axhline(0.05,  color='green', linestyle=':', alpha=0.7, label='positiv (0.05)')
ax.axhline(-0.05, color='red',   linestyle=':', alpha=0.7, label='negativ (-0.05)')
ax.axhline(0,     color='gray',  linestyle='--', linewidth=0.8)

ax.set_xticks(range(len(subs)))
ax.set_xticklabels(subs, rotation=20, ha='right')
ax.set_ylabel('Compound Score')
ax.set_title('F2: Sentiment-Verteilung nach Subreddit\n(Median-Linie schwarz, Punkte = einzelne Posts)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.savefig('../notebooks/analyse_f2_subreddits.png', dpi=120, bbox_inches='tight')
plt.show()
print('Gespeichert: notebooks/analyse_f2_subreddits.png')


---
## F3: Zeitliche Muster im Sentiment

In [ ]:
infer_category = market_metadata.infer_category

if 'created_utc' in posts_df.columns:
    df_time = posts_df.copy()
    df_time['created_utc'] = pd.to_datetime(df_time['created_utc'], errors='coerce')
    df_time = df_time.dropna(subset=['created_utc'])
    df_time['date']    = df_time['created_utc'].dt.date
    df_time['hour']    = df_time['created_utc'].dt.hour
    df_time['weekday'] = df_time['created_utc'].dt.day_name()

    if 'category' in df_time.columns and df_time['category'].notna().any():
        df_time['category'] = df_time['category'].fillna('Other')
    elif 'market_question' in df_time.columns:
        df_time['category'] = df_time['market_question'].apply(infer_category)
    else:
        df_time['category'] = 'Other'

    daily = df_time.groupby('date').agg(
        mean_compound=('compound', 'mean'),
        post_count=('compound', 'count')
    ).reset_index()

    cat_daily = (df_time
        .groupby(['date', 'category'])
        .agg(mean_compound=('compound', 'mean'),
             post_count=('compound', 'count'))
        .reset_index()
    )

    print(f"Zeitraum: {df_time['date'].min()} bis {df_time['date'].max()}")
    print(f"Tage mit Daten: {len(daily)}  |  Posts gesamt: {len(df_time)}")
    print("Kategorien:", df_time['category'].value_counts().to_dict())
else:
    print('Keine Zeitstempel in den Daten.')
    daily     = pd.DataFrame()
    cat_daily = pd.DataFrame()
    df_time   = pd.DataFrame()


In [ ]:
# F3: Zeitreihe nach Kategorie
if not cat_daily.empty:
    palette    = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22']
    categories = sorted(cat_daily['category'].unique())
    colors     = {cat: palette[i % len(palette)] for i, cat in enumerate(categories)}

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8),
                                    gridspec_kw={'height_ratios': [2.5, 1]})

    for cat in categories:
        grp = cat_daily[cat_daily['category'] == cat].sort_values('date')
        if len(grp) < 2:
            continue
        n_p = grp['post_count'].sum()
        ax1.plot(grp['date'], grp['mean_compound'],
                 marker='o', markersize=4, linewidth=1.5,
                 label=f'{cat} (n={n_p})', color=colors[cat], alpha=0.85)

    ax1.axhline(0,      color='gray',  lw=0.8, ls='--')
    ax1.axhline( 0.05,  color='green', lw=0.5, ls=':', alpha=0.6, label='positiv (0.05)')
    ax1.axhline(-0.05,  color='red',   lw=0.5, ls=':', alpha=0.6, label='negativ (-0.05)')
    ax1.set_ylabel('Oe Compound Score (RoBERTa)')
    ax1.set_title('F3: Sentiment-Zeitreihe nach Markt-Kategorie')
    ax1.legend(fontsize=8, ncol=2, loc='upper left')
    ax1.grid(True, alpha=0.18)
    plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')

    vol = df_time.groupby(['date', 'category'])['compound'].count().unstack(fill_value=0)
    col_colors = [colors.get(c, 'gray') for c in vol.columns]
    vol.plot(kind='bar', stacked=True, ax=ax2,
             color=col_colors, alpha=0.75, legend=False, width=0.85)
    ax2.set_xlabel('Datum')
    ax2.set_ylabel('Posts pro Tag')
    ax2.set_title('Post-Volumen nach Kategorie')
    n_bars = len(vol)
    step   = max(1, n_bars // 8)
    ax2.set_xticks(range(0, n_bars, step))
    ax2.set_xticklabels([str(d) for d in vol.index[::step]],
                        rotation=30, ha='right', fontsize=7)
    ax2.grid(True, alpha=0.15)

    n_total = len(posts_df)
    plt.suptitle('F3: Zeitliche Entwicklung des Reddit-Sentiments  |  ' + str(n_total) + ' Posts, 30 Maerkte',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../notebooks/analyse_f3_zeitverlauf.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Gespeichert: notebooks/analyse_f3_zeitverlauf.png')

elif not daily.empty:
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax1.plot(daily['date'], daily['mean_compound'],
             color='steelblue', marker='o', lw=1.5, ms=4)
    ax1.axhline(0, color='gray', ls='--', lw=0.8)
    ax1.fill_between(daily['date'], daily['mean_compound'], 0,
                     where=daily['mean_compound'] >= 0, alpha=0.15, color='green')
    ax1.fill_between(daily['date'], daily['mean_compound'], 0,
                     where=daily['mean_compound'] < 0,  alpha=0.15, color='red')
    ax2 = ax1.twinx()
    ax2.bar(daily['date'], daily['post_count'], alpha=0.2, color='gray')
    ax2.set_ylabel('Anzahl Posts', color='gray')
    ax1.set_ylabel('Oe Compound Score')
    ax1.set_title('F3: Sentiment ueber Zeit + Post-Volumen')
    plt.tight_layout()
    plt.savefig('../notebooks/analyse_f3_zeitverlauf.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Keine Zeitdaten verfuegbar.')


In [ ]:
# F3b: Sentiment nach Wochentag
if 'created_utc' in posts_df.columns:
    day_order  = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    day_labels = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
    day_stats  = df_time.groupby('weekday')['compound'].agg(['mean', 'count']).reindex(day_order).dropna()

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ['#2ecc71' if v >= 0.05 else '#e74c3c' if v <= -0.05 else '#f1c40f'
              for v in day_stats['mean']]
    ax.bar(day_labels[:len(day_stats)], day_stats['mean'],
           color=colors, edgecolor='white', alpha=0.9)

    for i, (_, row) in enumerate(day_stats.iterrows()):
        offset = 0.007 if row['mean'] >= 0 else -0.007
        va = 'bottom' if row['mean'] >= 0 else 'top'
        ax.text(i, row['mean'] + offset,
                f"{row['mean']:+.3f} (n={int(row['count'])})",
                ha='center', va=va, fontsize=8.5, fontweight='bold')

    ax.axhline(0,      color='black', linewidth=0.8)
    ax.axhline( 0.05,  color='green', lw=0.6, ls=':', alpha=0.7, label='positiv (0.05)')
    ax.axhline(-0.05,  color='red',   lw=0.6, ls=':', alpha=0.7, label='negativ (-0.05)')
    n_total = len(df_time)
    ax.set_title('F3b: Sentiment nach Wochentag  |  Oe RoBERTa-Score  |  ' + str(n_total) + ' Posts')
    ax.set_ylabel('Oe Compound Score')
    ax.set_xlabel('Wochentag')
    ax.legend(fontsize=8)
    ax.set_ylim(day_stats['mean'].min() - 0.08, day_stats['mean'].max() + 0.08)
    ax.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    plt.savefig('../notebooks/analyse_f3_weekday.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Gespeichert: notebooks/analyse_f3_weekday.png')
else:
    print('Keine Zeitstempel vorhanden.')


---
## F4: Stance-Korrelation ? Glauben Reddit-Nutzer, dass das Ereignis eintritt?

**Forschungsfrage F4:**  
Korreliert der Stance-Score staerker mit Polymarket-Wahrscheinlichkeiten als reines Sentiment?

**Methode:**  
Modell: `MoritzLaurer/deberta-v3-base-zeroshot-v2.0`  
Stance-Score: P(Hypothese wird unterstuetzt) - P(Hypothese wird abgelehnt), Wertebereich [-1, +1].  
Die finalen Scores werden mit `python scripts/add_stance_scores.py` in die CSVs geschrieben.


In [ ]:
# ── F4: Stance-Korrelation ────────────────────────────────────────────
import os, sys
sys.path.insert(0, '..')

BULK_CSV_F4 = os.path.join('..', 'data', 'correlation_pairs_bulk.csv')
if not os.path.exists(BULK_CSV_F4):
    BULK_CSV_F4 = os.path.join('data', 'correlation_pairs_bulk.csv')

try:
    df_f4 = pd.read_csv(BULK_CSV_F4)
except FileNotFoundError:
    df_f4 = pd.DataFrame()

if df_f4.empty or 'stance_score' not in df_f4.columns or df_f4['stance_score'].isna().all():
    print('stance_score nicht verfuegbar.')
    print('  -> python scripts/add_stance_scores.py ausfuehren, dann Notebook neu laden.')
else:
    valid = df_f4.dropna(subset=['stance_score', 'probability', 'adjusted_weighted'])

    if len(valid) < 3:
        print(f'Zu wenige Datenpunkte mit Stance-Score: {len(valid)}')
    else:
        r_stance, p_stance = stats.pearsonr(valid['probability'], valid['stance_score'])
        r_senti,  p_senti  = stats.pearsonr(valid['probability'], valid['adjusted_weighted'])

        print(f'F4 Ergebnisse (n={len(valid)} Maerkte):')
        print(f'  r(Stance)    = {r_stance:+.3f}  p={p_stance:.3f}')
        print(f'  r(Sentiment) = {r_senti:+.3f}  p={p_senti:.3f}')

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))

        # Links: Stance-Score vs. Polymarket-Wahrscheinlichkeit
        ax = axes[0]
        ax.scatter(valid['probability'], valid['stance_score'],
                   s=60, alpha=0.7, color='#2ecc71')
        m_st, b_st = np.polyfit(valid['probability'], valid['stance_score'], 1)
        x_line = np.linspace(valid['probability'].min(), valid['probability'].max(), 100)
        ax.plot(x_line, m_st * x_line + b_st, '--', color='#27ae60', lw=1.5)
        ax.set_xlabel('Polymarket-Wahrscheinlichkeit')
        ax.set_ylabel('Stance-Score')
        ax.set_title(f'Stance vs. Polymarket\nr = {r_stance:+.3f}  p = {p_stance:.3f}')
        ax.axhline(0, color='gray', lw=0.8, ls=':')
        ax.axvline(0.5, color='gray', lw=0.8, ls=':')

        # Rechts: Vergleich r(Sentiment) vs. r(Stance)
        ax2 = axes[1]
        labels_bar = ['Sentiment\n(adjusted weighted)', 'Stance\n(NLI)']
        r_vals     = [r_senti, r_stance]
        colors_bar = ['#3498db' if r >= 0 else '#e74c3c' for r in r_vals]
        bars = ax2.bar(labels_bar, r_vals, color=colors_bar,
                       width=0.4, edgecolor='white')
        ax2.axhline(0, color='black', lw=0.8)
        ax2.set_ylim(-1, 1)
        ax2.set_ylabel('Pearson r')
        ax2.set_title('Sentiment vs. Stance: Pearson r')
        for bar, r_val in zip(bars, r_vals):
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                r_val + (0.03 if r_val >= 0 else -0.07),
                f'{r_val:+.3f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.savefig('analyse_f4_stance.png', dpi=120, bbox_inches='tight')
        plt.show()
        print('Gespeichert: analyse_f4_stance.png')


---
## Zusammenfassung & Fazit

In [ ]:
print('=' * 60)
print('ZUSAMMENFASSUNG')
print('=' * 60)

total = len(posts_df)
pos = (posts_df['sentiment_label'] == 'positive').sum()
neg = (posts_df['sentiment_label'] == 'negative').sum()
neu = (posts_df['sentiment_label'] == 'neutral').sum()

print(f'\nDatensatz:')
print(f'  Posts gesamt:    {total}')
print(f'  Positiv:         {pos} ({pos/total*100:.1f}%)')
print(f'  Neutral:         {neu} ({neu/total*100:.1f}%)')
print(f'  Negativ:         {neg} ({neg/total*100:.1f}%)')
print(f'  Ø Compound:      {posts_df["compound"].mean():+.4f}')

print(f'\nF1 – Sentiment vs. Polymarket:')
if not markets_df.empty and 'probability' in markets_df.columns:
    print(f'  Reddit Sentiment (normalisiert): {(posts_df["compound"].mean()+1)/2:.3f}')
    print(f'  Polymarket Ø Wahrscheinlichkeit: {markets_df["probability"].mean():.3f}')
else:
    print('  Polymarket-Daten nicht verfügbar')

print(f'\nF2 – Subreddit-Unterschiede:')
if len(posts_df['subreddit'].unique()) > 1:
    best = sub_stats['mean'].idxmax()
    worst = sub_stats['mean'].idxmin()
    print(f'  Positivstes Subreddit:  {best} ({sub_stats.loc[best,"mean"]:+.3f})')
    print(f'  Negativstes Subreddit:  {worst} ({sub_stats.loc[worst,"mean"]:+.3f})')

print(f'\nF3 – Zeitliche Muster:')
if not daily.empty:
    best_day = daily.loc[daily['mean_compound'].idxmax(), 'date']
    worst_day = daily.loc[daily['mean_compound'].idxmin(), 'date']
    print(f'  Positivster Tag:  {best_day} ({daily["mean_compound"].max():+.3f})')
    print(f'  Negativster Tag:  {worst_day} ({daily["mean_compound"].min():+.3f})')
else:
    print('  Keine Zeitdaten verfügbar')

## Fazit und Diskussion

### Ergebnisse des finalen Runs

- **F1 ? Korrelation Polymarket ? Reddit-Sentiment:**
  Der finale Live-Bulk-Run enthaelt **29 auswertbare Maerkte** und **725 Reddit-Posts**.
  Fuer `adjusted_weighted` ergibt sich Pearson r = **+0.0791** (p = 0.6833) und
  Spearman rho = **+0.1508** (p = 0.4350). Damit gibt es keinen statistisch
  belastbaren linearen Zusammenhang im Gesamtdatensatz.

- **Polarity-Korrektur:**
  Negativ gerahmte Fragen werden mit `src/market_metadata.py` erkannt und im Vorzeichen
  korrigiert. Dadurch wird positives Reddit-Sentiment bei Fragen wie Rezession, Krieg
  oder Verurteilung nicht faelschlich als Zustimmung zum negativen Ereignis interpretiert.

- **Richtungsuebereinstimmung:**
  Reddit-Signal und Marktrichtung stimmen in **44.8%** der Maerkte ueberein
  (**13 von 29**). Das spricht im aktualisierten Run nicht fuer Prognosekraft und wird nicht kausal
  oder prognostisch ueberinterpretiert.

- **F1b ? Kategorie-Korrelation:**
  Die finale Taxonomie umfasst Sports, Legal, Geopolitics, Entertainment, Politics,
  Crypto und Other. Einzelne Kategorieergebnisse sind wegen kleiner Gruppen explorativ.

- **F2 ? Subreddit-Unterschiede:**
  Der Kruskal-Wallis-Test ist deutlich signifikant (**H = 108.08, p < 0.001**).
  Finanznahe Subreddits sind im Mittel positiver als Politik- und News-Subreddits.

- **F3 ? Zeitliche Muster:**
  Der Datensatz deckt Reddit-Zeitstempel von **2010-09-02 bis 2026-05-20** ab,
  verteilt auf **333 Tage** mit Daten. Fuer die Berichtsgrafik wird der Zeitraum
  aus Lesbarkeitsgruenden auf Posts ab **2025-01-01** gekuerzt (**566 Posts**,
  **144 Tage**). Da Polymarket nur als aktueller Snapshot vorliegt, ist keine
  robuste Lag-Analyse moeglich.

- **F4 ? Stance Detection:**
  Stance Detection wurde nach dem Bulk-Run mit `scripts/add_stance_scores.py` berechnet.
  Pearson r(Stance) = **-0.0855** (p = 0.6591), Spearman rho = **-0.1123** (p = 0.5618).
  Stance ist methodisch naeher an der Marktfrage, liefert in dieser Stichprobe aber ebenfalls
  keinen signifikanten Zusammenhang.

---

### Methodische Einschraenkungen

1. **Stichprobengroesse:** 29 Maerkte sind fuer eine explorative Analyse geeignet, aber zu klein fuer robuste Inferenz.
2. **Reddit-Bias:** Reddit ist kein repraesentatives Abbild der Gesamtbevoelkerung oder aller Marktteilnehmer.
3. **Sentiment ? Stance:** RoBERTa misst emotionalen Ton, nicht zwingend Zustimmung zur Marktfrage.
4. **Querschnitt statt Panel:** Echte Lead-/Lag-Aussagen erfordern wiederholte Marktpreis-Snapshots.
5. **Kategorie-Inferenz:** Kategorien werden per Keyword-Taxonomie approximiert und koennen Randfaelle falsch klassifizieren.
